# 医学图像分割项目 - 完整流水线

本notebook整合了医学图像分割项目的完整流水线，包括：
1. 📥 数据集下载
2. 🔄 数据预处理  
3. 🚀 模型训练

## 项目概述
- **任务**: 基于语言引导的医学图像分割
- **数据集**: QaTA-dataset (COVID-19医学图像)
- **模型**: 使用BiomedVLP-CXR-BERT进行文本理解
- **框架**: PyTorch Lightning


In [5]:
# ============================================================
# 🔧 环境设置和依赖导入
# ============================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 设置环境变量
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 基础库
import pandas as pd
import torch
import numpy as np
from pathlib import Path

# 深度学习框架
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler

# 模型和数据处理
from transformers import AutoTokenizer, T5ForConditionalGeneration
from torchmetrics import Accuracy, Dice
from torchmetrics.classification import BinaryJaccardIndex
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# 数据下载
import gdown
import zipfile

# 进度条
from tqdm import tqdm

# 设置多进程策略
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

print("✅ 环境设置完成")
print(f"🔥 CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")


✅ 环境设置完成
🔥 CUDA可用: True
🎮 GPU设备: NVIDIA GeForce RTX 5090
💾 GPU内存: 31.4GB


In [6]:
# ============================================================
# ⚙️ 配置参数设置
# ============================================================

class Config:
    """项目配置类"""
    
    # 训练参数
    train_batch_size = 32
    valid_batch_size = 8
    lr = 0.0003
    min_epochs = 15
    max_epochs = 15
    patience = 10000
    device = 1
    image_size = [224, 224]
    
    # 模型参数
    bert_type = "microsoft/BiomedVLP-CXR-BERT-specialized"
    vision_type = None
    project_dim = 768
    
    # 数据路径
    data_dir = "./data/QaTA-dataset"
    train_csv_path = "./data/QaTA-dataset/Train_text_for_Covid19_processed.xlsx"
    train_root_path = "./data/QaTA-dataset/Train"
    test_csv_path = "./data/QaTA-dataset/Test_text_for_Covid19_processed.xlsx"
    test_root_path = "./data/QaTA-dataset/Test"
    
    # 模型保存
    model_save_path = "./save_model"
    model_save_filename = "medseg"
    
    # 数据下载
    dataset_url = "https://drive.google.com/uc?id=13DFm9p_aw_DRI2cqpVfCvAjMkdGzLpHT"
    dataset_zip = "Qata_dataset.zip"

# 创建配置实例
cfg = Config()
print("✅ 配置参数加载完成")
print(f"📊 批次大小: 训练={cfg.train_batch_size}, 验证={cfg.valid_batch_size}")
print(f"🎯 训练轮数: {cfg.min_epochs}-{cfg.max_epochs}")
print(f"🤖 模型类型: {cfg.bert_type}")


✅ 配置参数加载完成
📊 批次大小: 训练=32, 验证=8
🎯 训练轮数: 15-15
🤖 模型类型: microsoft/BiomedVLP-CXR-BERT-specialized


# 📥 第一步：数据集下载

从Google Drive下载QaTA数据集并解压到本地目录。


In [3]:
# ============================================================
# 📥 数据集下载函数
# ============================================================

def download_dataset(url, output_file, extract_dir="data"):
    """
    下载并解压数据集
    
    Args:
        url: Google Drive下载链接
        output_file: 输出文件名
        extract_dir: 解压目录
    """
    print(f"🔄 开始下载数据集...")
    print(f"📁 URL: {url}")
    print(f"💾 保存到: {output_file}")
    
    # 检查文件是否已存在
    if os.path.exists(output_file):
        print(f"⚠️  文件 {output_file} 已存在，跳过下载")
    else:
        # 下载文件
        gdown.download(url, output_file, quiet=False)
        print(f"✅ 下载完成: {output_file}")
    
    # 检查解压目录是否存在
    if os.path.exists(extract_dir) and os.listdir(extract_dir):
        print(f"⚠️  目录 {extract_dir} 已存在且非空，跳过解压")
    else:
        # 解压文件
        print(f"🗂️  开始解压到 {extract_dir} 目录...")
        with zipfile.ZipFile(output_file, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"✅ 解压完成")
    
    # 显示数据集结构
    data_path = os.path.join(extract_dir, "QaTA-dataset")
    if os.path.exists(data_path):
        print(f"\n📊 数据集结构:")
        for item in os.listdir(data_path):
            item_path = os.path.join(data_path, item)
            if os.path.isdir(item_path):
                file_count = len(os.listdir(item_path)) if os.path.exists(item_path) else 0
                print(f"  📁 {item}/ ({file_count} 个文件)")
            else:
                print(f"  📄 {item}")

print("✅ 数据集下载函数定义完成")


✅ 数据集下载函数定义完成


In [7]:
# ============================================================
# 🚀 执行数据集下载
# ============================================================

# 开始下载数据集
download_dataset(
    url=cfg.dataset_url,
    output_file=cfg.dataset_zip,
    extract_dir="data"
)

print("\n🎉 数据集下载阶段完成！")


NameError: name 'download_dataset' is not defined

# 🔄 第二步：数据预处理

使用T5模型对医学文本描述进行短语分割处理，提高文本的语义理解质量。


In [3]:
# ============================================================
# 🔄 数据预处理 - PhraseSplitter类和函数
# ============================================================

# 定义分隔符
SENT_PHRASES_SEP = "|||"

class PhraseSplitter:
    def __init__(self, t5_name="google/flan-t5-base"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        torch_dtype = torch.bfloat16 if (self.device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(t5_name)
        self.model = T5ForConditionalGeneration.from_pretrained(
            t5_name, torch_dtype=torch_dtype
        ).to(self.device).eval()

        self.prompt_prefix = "Split the sentence into separate meaningful medical phrases, "
        self.gen_kwargs = dict(num_beams=1, do_sample=False, max_new_tokens=24, use_cache=True)

    @torch.no_grad()
    def split_batch(self, sentences):
        prompts = [self.prompt_prefix + s for s in sentences]
        enc = self.tokenizer(prompts, padding=True, truncation=True, max_length=64, return_tensors="pt").to(self.device)
        ctx = (torch.autocast("cuda", dtype=self.model.dtype)
               if self.device == "cuda" and self.model.dtype in (torch.bfloat16, torch.float16)
               else torch.cuda.amp.autocast(enabled=False))
        with ctx:
            out = self.model.generate(**enc, **self.gen_kwargs)
        texts = self.tokenizer.batch_decode(out, skip_special_tokens=True)

        # 清洗成短语列表
        phrase_strs = []
        for t in texts:
            parts = [p.strip() for p in t.replace(" and ", ", ").split(",")]
            parts = [p for p in parts if p]
            phrase_strs.append(", ".join(parts))
        return phrase_strs

def process_xlsx(in_xlsx, out_xlsx, batch_size=64, t5_name="google/flan-t5-base"):
    print(f"Processing {in_xlsx} -> {out_xlsx}")

    df = pd.read_excel(in_xlsx)
    assert {"Image", "Description"}.issubset(df.columns), "需要列名：Image, Description"

    splitter = PhraseSplitter(t5_name)
    descs = df["Description"].astype(str).tolist()

    out_descs = []
    for i in tqdm(range(0, len(descs), batch_size), ncols=100):
        batch_sent = descs[i:i+batch_size]
        batch_phr_str = splitter.split_batch(batch_sent)
        for sent, phr in zip(batch_sent, batch_phr_str):
            if phr:
                out_descs.append(f"{sent.strip()} {SENT_PHRASES_SEP} {phr}")
            else:
                out_descs.append(sent.strip())

    out_df = pd.DataFrame({"Image": df["Image"].tolist(), "Description": out_descs})
    out_df.to_excel(out_xlsx, index=False)
    print(f"✅ Done. Saved to {out_xlsx}")

print("✅ 数据预处理函数定义完成")


✅ 数据预处理函数定义完成


In [8]:
# ============================================================
# 🚀 执行数据预处理
# ============================================================

# 自动处理Train/Test两个文件
data_dir = cfg.data_dir
train_xlsx = os.path.join(data_dir, "Train_text_for_Covid19.xlsx")
test_xlsx = os.path.join(data_dir, "Test_text_for_Covid19.xlsx")

process_xlsx(train_xlsx, os.path.join(data_dir, "Train_text_for_Covid19_processed.xlsx"))
process_xlsx(test_xlsx, os.path.join(data_dir, "Test_text_for_Covid19_processed.xlsx"))

print("\n🎉 数据预处理阶段完成！")


NameError: name 'process_xlsx' is not defined

# 🚀 第三步：模型训练

使用PyTorch Lightning框架训练语言引导的医学图像分割模型。


In [3]:
# ============================================================
# 🏗️ 导入训练相关模块
# ============================================================

# 导入项目中的自定义模块
# sys.path.append('/share_disk/yyl_test/NoteBook')

from utils.dataset import QaTa
from engine.wrapper import LanGuideMedSegWrapper

print("✅ 训练模块导入完成")


✅ 训练模块导入完成


In [ ]:
# ============================================================
# 🚀 执行模型训练
# ============================================================

print("cuda:", torch.cuda.is_available())

# 创建数据集
ds_train = QaTa(csv_path=cfg.train_csv_path,
                root_path=cfg.train_root_path,
                tokenizer=cfg.bert_type,
                image_size=cfg.image_size,
                mode='train')

ds_valid = QaTa(csv_path=cfg.train_csv_path,
                root_path=cfg.train_root_path,
                tokenizer=cfg.bert_type,
                image_size=cfg.image_size,
                mode='valid')

# 创建数据加载器
dl_train = DataLoader(ds_train, batch_size=cfg.train_batch_size, shuffle=True, num_workers=8)
dl_valid = DataLoader(ds_valid, batch_size=cfg.valid_batch_size, shuffle=False, num_workers=8)

# 创建模型
model = LanGuideMedSegWrapper(cfg)

# 设置回调函数
model_ckpt = ModelCheckpoint(
    dirpath=cfg.model_save_path,
    filename=cfg.model_save_filename,
    monitor='val_loss',
    save_top_k=1,
    mode='min',
    verbose=True,
)

early_stopping = EarlyStopping(monitor='val_loss',
                        patience=cfg.patience,
                        mode='min')

# 设置训练器
trainer = pl.Trainer(logger=True,
                    min_epochs=cfg.min_epochs, max_epochs=cfg.max_epochs,
                    accelerator='gpu',
                    devices=cfg.device,
                    callbacks=[model_ckpt, early_stopping],
                    enable_progress_bar=False,
                    )

# 开始训练
print('start training')
trainer.fit(model, dl_train, dl_valid)
print('done training')

print("\n🎉 训练阶段完成！")


cuda: True
<vig_pytorch.pyramid_vig.pvig_ti_224_gelu.<locals>.OptInit object at 0x73a8cccae2d0>
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
9
using relative_pos
Missing: 0 []
Unexpected: 0 []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


using final model
using final model
using final model
using final model
start training


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type           | Params
-------------------------------------------------
0 | model         | LanGuideMedSeg | 127 M 
1 | loss_fn       | DiceCELoss     | 0     
2 | train_metrics | ModuleDict     | 0     
3 | val_metrics   | ModuleDict     | 0     
4 | test_metrics  | ModuleDict     | 0     
-------------------------------------------------
15.6 M    Trainable params
111 M     Non-trainable params
127 M     Total params
508.196   Total estimated model params size (MB)



================================================================================2025-09-01 10:55:54
{'epoch': 0, 'val_loss': 1.832157850265503, 'val_acc': 0.1352177858352661, 'val_dice': 0.2142217606306076, 'val_MIoU': 0.11995989084243774}


...

<<<<<< reach best val_loss : 0.8796741366386414 >>>>>>
Epoch 13, global step 2506: 'val_loss' reached 0.87967 (best 0.87967), saving model to './save_model/medseg.ckpt' as top 1



================================================================================2025-09-01 11:06:34
{'epoch': 13, 'val_loss': 0.8796741366386414, 'val_acc': 0.9771623015403748, 'val_dice': 0.8756304383277893, 'val_MIoU': 0.7787745594978333}
{'epoch': 13, 'train_loss': 0.7688126564025879, 'train_acc': 0.9788570404052734, 'train_dice': 0.9193784594535828, 'train_MIoU': 0.8507867455482483}


Epoch 14, global step 2685: 'val_loss' was not in top 1
`Trainer.fit` stopped: `max_epochs=15` reached.



================================================================================2025-09-01 11:07:20
{'epoch': 14, 'val_loss': 0.8831965327262878, 'val_acc': 0.9773354530334473, 'val_dice': 0.8745559453964233, 'val_MIoU': 0.7770763039588928}
{'epoch': 14, 'train_loss': 0.7665284276008606, 'train_acc': 0.9791762232780457, 'train_dice': 0.9205962419509888, 'train_MIoU': 0.852874755859375}
done training

🎉 训练阶段完成！


: 